# Lab 04 · ReAct: choose a tool, then decide

**Prepared by Mustafa Assaf** · 5–10 minute example

## Let the model choose the next step

**So far:** in Lab 03 our code called the tools in a fixed order.

**New:** an *agent loop*. The model chooses an action, our code runs it, and the result (the *observation*) goes into the next prompt. This act-then-observe cycle is the core of the ReAct pattern.

Our code stays in control. It accepts only known actions, refuses to finish before the metrics are read, and stops after two turns. The tool is a plain Python copy of Lab 03's `read_metrics`, so the loop is easy to follow.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, save `FREEINFERENCE_API_KEY` in Colab Secrets and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/aebb52597ab982f0b8b2f0832815b73dd6aca416.zip"
from coursekit import make_ask
MODE = "practice"
ask = make_ask(mode=MODE)

## Step 1 · Run a bounded action loop

On each turn, the prompt lists the allowed actions and the latest observation. The code then runs the chosen action or stops. In practice mode, the scripted agent reads the metrics on turn 1 and finishes on turn 2.

In [ ]:
def read_metrics():
    return {"team_size": 3, "independent_deploys": False}
observation = None
for turn in range(1, 3):  # hard limit: at most two turns
    prompt = ("Actions: read_metrics, finish. Finish only after reading metrics.\n"
              f"Observation: {observation}\nReply with one action name.")
    action = ask(prompt, example="finish" if observation else "read_metrics").strip()
    print(f"Turn {turn}: {action}")
    if action == "finish" and observation:
        break
    if action != "read_metrics":
        raise ValueError(f"Action not allowed now: {action}")
    observation = read_metrics()
    print("Observation:", observation)
assert action == "finish", "Turn limit reached before the model finished."

## Step 2 · Decide from the observation

The final prompt combines the policy with what the agent observed.

In [ ]:
policy = "Choose microservices only if team_size >= 8 AND independent_deploys is true; otherwise choose modular monolith."
decision = ask(f"{policy} Metrics: {observation}. Return only the choice.", example="modular monolith").strip().lower()
print("Decision:", decision)
assert decision == "modular monolith", "Check the observation and the policy."

## What you should see

`Turn 1: read_metrics`, the observation, `Turn 2: finish`, then `Decision: modular monolith`. Three problems fail loudly instead of passing silently: an unknown action, finishing before reading the metrics, and running out of turns.

## Try it

Make `read_metrics` return `{"team_size": 8, "independent_deploys": True}` and predict the decision. In practice mode, update the scripted decision and the final `assert` to match. In live mode, check whether the model follows the new observation.

## Takeaway

Not every task needs an agent. A fixed *chain*, such as "always read, then decide", is simpler and more predictable. A loop helps when the next step depends on what was just observed. More model calls do not improve quality on their own.

## Read further

- [ReAct](https://arxiv.org/abs/2210.03629)
- [Workflows and agentic patterns](https://www.anthropic.com/research/building-effective-agents)